# Stroke Prevention Demo Baseline Model

DP1 Contributor: Erik Bergmark (erikcb2)

DP2 Contributor: Joshua Lee (jcl12)

Consolidated by: Daryl Okeke (dokek2)

## Purpose
Train a baseline logistic regression model to predict `stroke`.

Uses a standardized train test split and evaluates on the test set only.

Saves metrics to `reports/baseline_metrics.md`.

## How to run
Run all cells from top to bottom.


## 1. Config

This is the main place to tweak the baseline.

If you are not sure what a parameter does, you can search the scikit learn docs for `LogisticRegression`.


### What to tweak here
This section controls the baseline without needing to read the rest of the notebook.

- TEST_SIZE: How much data we hold out for testing. 0.2 means 20 percent test, 80 percent train.
- RANDOM_STATE: The seed so everyone gets the same split and the same results.
- CLASS_WEIGHT: Helps when strokes are rare. Set to "balanced" if the model predicts all zeros.
- MAX_ITER: How long logistic regression is allowed to try before stopping. Increase if you see a convergence warning.
- SOLVER: The optimization method. "liblinear" is a safe default for small to medium data.

In [43]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Try "balanced" later to address class imbalance
CLASS_WEIGHT = "balanced"

# Logistic regression settings
MAX_ITER = 1000
SOLVER = "liblinear"


## 2. Imports


In [44]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
)


## 3. Load data

Expected file path: `data/raw/stroke_data.csv` at the repo root.


### What this does
This loads the dataset from the repo so everyone runs the same file.

- ROOT: The project folder. If you run the notebook from notebooks/, it moves up one folder.
- DATA_PATH: Where the CSV should be located in the repo.
- df: The full dataset as a table.
If this cell errors, the dataset file is missing or the path is wrong.

In [45]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "raw" / "stroke_data.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset at {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df.head()


,stroke,gender,age,Race,Marital status,alcohol,smoke,sleep disorder,Health Insurance,General health condition,...,energy,protein,Carbohydrate,Dietary fiber,Total fat,Total saturated fatty acids,Total monounsaturated fatty acids,Total polyunsaturated fatty acids,Potassium,Sodium
0,0,2,2,5,1,0,0,2,2,3,...,1598,62.78,192.19,10.0,65.64,25.112,24.090,8.543,2887,2969
1,0,2,2,1,1,0,0,1,2,3,...,1547,45.35,256.02,17.0,42.56,13.423,15.389,10.613,2058,2091
2,1,1,2,3,1,1,1,2,1,3,...,2466,81.56,254.49,13.0,103.32,43.295,36.727,15.366,3117,5233
3,0,2,3,3,1,1,1,2,1,4,...,1605,70.99,143.37,10.0,81.60,24.527,30.567,18.174,1766,3706
4,0,1,1,4,1,0,0,2,1,2,...,1818,74.75,229.45,14.2,67.49,26.030,24.837,10.533,1842,2461


## 4. Define features and label


### What this does
This splits the dataset into inputs and the thing we are trying to predict.

- y: The label we want to predict. Here it is df["stroke"].
- X: The input features. This is every column except "stroke".

Label counts and proportions show class imbalance.
If stroke is rare, the model can look good on accuracy while being useless.

### Common tweak
If we discover leakage columns, drop them here before the train test split.

Example:
X = X.drop(columns=["leak_col_1", "leak_col_2"])

In [46]:
if "stroke" not in df.columns:
    raise ValueError("Dataset must contain a 'stroke' column.")

# Drop leakage columns
df = df.drop(columns=["General health condition", "depression", "Minutes sedentary activity"])

df.head()

y = df["stroke"]
X = df.drop(columns=["stroke"])

print("Label counts")
print(y.value_counts(dropna=False))
print("\nLabel proportions")
print(y.value_counts(normalize=True, dropna=False))


Label counts
stroke
0    4241
1     362
Name: count, dtype: int64

Label proportions
stroke
0    0.921356
1    0.078644
Name: proportion, dtype: float64


## 5. Train test split

### What this does
This splits data into train and test.

- Train set: what the model learns from.
- Test set: what we use to score the model on unseen data.

We try stratify first.
Stratify keeps the same stroke ratio in train and test.
If stratify fails, we fall back to no stratify and keep going.


In [47]:
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    stratify_used = "yes"
    stratify_note = ""
except ValueError as e:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=None,
    )
    stratify_used = "no"
    stratify_note = str(e)

print(f"Split standardized: test_size={TEST_SIZE}, seed={RANDOM_STATE}, stratify={stratify_used}")
if stratify_note:
    print("Stratify note")
    print(stratify_note)


Split standardized: test_size=0.2, seed=42, stratify=yes


## 6. Preprocessing

### What this does
This prepares the data so the model can use it.

- cat_cols: Columns that are text categories like gender or race.
- num_cols: Columns that are numbers like age or BMI.
- OneHotEncoder: Converts each category into columns of 0 and 1 so the model can use them.
- handle_unknown="ignore": If a new category shows up in test data, it will not crash.

**We likely will not need to change this section unless we change how we handle categories.**


In [48]:
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Number of categorical columns", len(cat_cols))
print("Number of numeric columns", len(num_cols))

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)


Number of categorical columns 0
Number of numeric columns 32


## 7. Model

### What to tweak here
This is the main modeling block.

- LogisticRegression: A simple baseline model.
- class_weight: Set to "balanced" if it predicts all zeros and recall is 0.
- max_iter: Increase if training warns about not converging.
- solver: Usually keep "liblinear" for simplicity.

Common tweaks:
- Try class_weight="balanced"
- Try adding C=0.5 or C=2.0 to change regularization strength


In [49]:
model = LogisticRegression(
    max_iter=MAX_ITER,
    solver=SOLVER,
    class_weight=CLASS_WEIGHT,
    random_state=RANDOM_STATE,
)

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", model),
])


## 8. Train


In [50]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  []),
                                                 ('num', 'passthrough',
                                                  ['gender', 'age', 'Race',
                                                   'Marital status', 'alcohol ',
                                                   'smoke', 'sleep disorder',
                                                   'Health Insurance',
                                                   'sleep time', 'diabetes',
                                                   'hypertension',
                                                   'high cholesterol',
                                                   'Coronary Heart Disease',
                                                   'Body Mass Index',
                                                   'Waist Circumference',
                                                   'Systolic b...
                                                   'High-density lipoprotein',
                                                   'Triglyceride',
                                                   'Low-density lipoprotein',
                                                   'Fasting Glucose',
                                                   'Glycohemoglobin', 'energy',
                                                   'protein', 'Carbohydrate',
                                                   'Dietary fiber', 'Total fat',
                                                   'Total saturated fatty '
                                                   'acids',
                                                   'Total monounsaturated '
                                                   'fatty acids',
                                                   'Total polyunsaturated '
                                                   'fatty acids', ...])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42, solver='liblinear'))])

## 9. Evaluate on test set only

### What this does
This scores the model on the test set only.

- y_pred: The final 0 or 1 predictions using the default threshold of 0.5.
- y_prob: The predicted probability of stroke being 1. This is used for ROC AUC.
- Accuracy: Percent correct overall.
- Precision: When we predict stroke, how often we are right.
- Recall: Of the true stroke cases, how many we catch.
- ROC AUC: How well the model ranks stroke cases higher than non stroke across all thresholds.
- Confusion matrix: A table of TN FP FN TP.

If recall is 0, it usually means the model predicted all zeros.
That is common when stroke is rare.


In [51]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

try:
    roc_auc = roc_auc_score(y_test, y_prob)
except Exception:
    roc_auc = None

cm = confusion_matrix(y_test, y_pred)

print("Accuracy", accuracy)
print("Precision", precision)
print("Recall", recall)
print("ROC AUC", roc_auc)
print("Confusion matrix")
print(cm)


Accuracy 0.6547231270358306
Precision 0.11075949367088607
Recall 0.4861111111111111
ROC AUC 0.6101295641931685
Confusion matrix
[[568 281]
 [ 37  35]]


/Users/darylokeke/Documents/WORK/HAS-StrokePrev/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/darylokeke/Documents/WORK/HAS-StrokePrev/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/darylokeke/Documents/WORK/HAS-StrokePrev/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/darylokeke/Documents/WORK/HAS-StrokePrev/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/darylokeke/Documents/WORK/HAS-StrokePrev/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Us

## 10. Save report

Writes to `reports/baseline_metrics.md` at the repo root. Change the report_path naming to your ticket so we know what was changed to get to that metric.

If you add new metrics later, also add them to this report so the repo stays consistent.


In [ ]:
reports_dir = ROOT / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
report_path = reports_dir / "DPbaseline_metrics.md" # Change this to make a markdown of your ticket-specific metrics (Ex. DP4_metrics.md)

with open(report_path, "w", encoding="utf-8") as f:
    f.write("# Baseline Metrics\n\n")
    f.write("## Model\n")
    f.write("Logistic Regression\n\n")

    f.write("## Split\n")
    f.write(f"test_size: {TEST_SIZE}\n")
    f.write(f"random_state: {RANDOM_STATE}\n")
    f.write(f"stratify: {stratify_used}\n")
    if stratify_note:
        f.write("stratify_note:\n")
        f.write(f"{stratify_note}\n")
    f.write("\n")

    f.write("## Label balance\n")
    f.write("Counts\n")
    f.write(y.value_counts(dropna=False).to_string())
    f.write("\n\n")

    f.write("## Metrics on test set\n")
    f.write(f"Accuracy: {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall: {recall:.4f}\n")
    f.write(f"ROC AUC: {roc_auc if roc_auc is not None else 'N A'}\n\n")

    f.write("## Confusion matrix on test set\n")
    f.write("Format is [[TN FP]\n")
    f.write("           [FN TP]]\n\n")
    f.write(str(cm))
    f.write("\n")

print(f"Saved report to {report_path}")
print(
    f"Baseline standardized: test_size={TEST_SIZE}, seed={RANDOM_STATE}, stratify={stratify_used}, metrics saved in reports/baseline_metrics.md"
)


Saved report to /Users/darylokeke/Documents/WORK/HAS-StrokePrev/stroke-prevention-demo/reports/baseline_metrics.md
Baseline standardized: test_size=0.2, seed=42, stratify=yes, metrics saved in reports/baseline_metrics.md


## Common tweaks:

1. Class imbalance: set `CLASS_WEIGHT = "balanced"`
2. Drop leakage columns: `X = X.drop(columns=[...])` before the split
3. Threshold tuning: pick a threshold and convert probabilities to 0 or 1
